# Experiments

- **Exp 1**: Reward Function Comparison (Sortino vs Sharpe vs Raw)
- **Exp 2**: PPO vs Baselines (Equal-Weight, B&H BTC, MktCap-Weight)
- **Exp 3**: MLP vs LSTM Policy Comparison

In [1]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath(".."))

from src.utils.data_utils import load_all_data, get_split, WALK_FORWARD_SPLITS, DEFAULT_TICKERS
from src.evaluation.evaluate_portfolio import (
    run_backtest, run_equal_weight_baseline, run_bnh_baseline, run_mcap_baseline,
    plot_nav_comparison, generate_report_table,
)

# Project root = parent of experiments/
_ROOT = os.path.abspath(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))     if "__file__" in dir() else os.path.abspath("..")     if os.path.isdir("models") == False else os.path.abspath(".")
# Simpler: anchor on models/ folder
_ROOT = os.path.abspath(".")
while not os.path.isdir(os.path.join(_ROOT, "models")) and _ROOT != os.path.dirname(_ROOT):
    _ROOT = os.path.dirname(_ROOT)
RESULTS_DIR     = os.path.join(_ROOT, "experiments", "results")
MODELS_DIR_MLP  = os.path.join(_ROOT, "models")
MODELS_DIR_LSTM = os.path.join(_ROOT, "models_lstm")
print(f"Project root: {_ROOT}")
SEEDS           = [42, 123, 777]
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Loading data...")
all_data = load_all_data(normalize=True, tickers=DEFAULT_TICKERS)
print("Done.")

Project root: e:\FPTU\FPTU_SE8\REL301m
Loading data...
[data_utils] BNB: dropping 1 day(s) with |return|>50%: ['2021-02-19']
[data_utils] Loaded 3 assets, 1977 aligned days (2020-01-01 to 2025-05-31)
[data_utils] Assets: ['BTC', 'ETH', 'BNB']
Done.


In [2]:
# Helper functions

def find_mlp_model(reward_type, win_id, seed):
    for cash_tag in ["_cash", ""]:
        name = f"ppo_mlp_{reward_type}_w{win_id}_s{seed}{cash_tag}"
        best = os.path.join(MODELS_DIR_MLP, name, "best_model.zip")
        final = os.path.join(MODELS_DIR_MLP, f"{name}_final.zip")
        if os.path.exists(best):
            return best
        if os.path.exists(final):
            return final
    return None


def find_lstm_model(reward_type, win_id, seed):
    name = f"ppo_lstm_{reward_type}_w{win_id}_s{seed}_cash"
    best = os.path.join(MODELS_DIR_LSTM, name, "best_model.zip")
    final = os.path.join(MODELS_DIR_LSTM, f"{name}_final.zip")
    if os.path.exists(best):
        return best
    if os.path.exists(final):
        return final
    return None


def avg_seeds(results_list):
    if not results_list:
        return {}
    keys = list(results_list[0].keys())
    return {k: round(float(np.mean([r[k] for r in results_list if k in r])), 3) for k in keys}

## Experiment 1: Reward Function Comparison
Sortino vs Sharpe vs Raw reward -- same PPO-MLP, same hyperparams

In [3]:
REWARD_TYPES  = ["sortino_style", "sharpe_style", "raw"]
REWARD_LABELS = {"sortino_style": "PPO-Sortino", "sharpe_style": "PPO-Sharpe", "raw": "PPO-Raw"}

exp1_all_metrics = {r: [] for r in REWARD_TYPES}

for split_cfg in WALK_FORWARD_SPLITS[:2]:
    win_id = split_cfg["id"]
    print(f"\n{'='*60}")
    print(f"Window {win_id}: test {split_cfg['test_start']} to {split_cfg['test_end']}")
    print(f"{'='*60}")

    _, test_data = get_split(all_data, split_cfg)
    test_index = next(iter(test_data.values())).index

    nav_dict = {}
    metrics_dict = {}

    for reward_type in REWARD_TYPES:
        label = REWARD_LABELS[reward_type]
        seed_metrics = []
        seed_navs = []

        for seed in SEEDS:
            mp = find_mlp_model(reward_type, win_id, seed)
            if mp is None:
                print(f"  [WARN] {label} seed={seed} not found")
                continue
            nav, metrics, _ = run_backtest(
                model_path=mp, test_data=test_data,
                reward_type=reward_type, use_cash=True, lambda_risk=0.3,
            )
            seed_metrics.append(metrics)
            seed_navs.append(nav)
            print(f"  {label} s{seed}: Return={metrics['total_return']:+.1f}%  "
                  f"Sortino={metrics['sortino']:.3f}  MDD={metrics['max_drawdown']:.1f}%")

        if seed_metrics:
            avg = avg_seeds(seed_metrics)
            metrics_dict[label] = avg
            exp1_all_metrics[reward_type].append(avg)
            min_len = min(len(n) for n in seed_navs)
            nav_dict[label] = np.stack([n[:min_len] for n in seed_navs]).mean(axis=0)

    # Equal-Weight baseline
    ew_nav, ew_met = run_equal_weight_baseline(test_data)
    nav_dict["Equal-Weight"] = ew_nav
    metrics_dict["Equal-Weight"] = ew_met

    # Plot
    plot_nav_comparison(
        nav_dict=nav_dict,
        title=f"Exp1: Reward Comparison - W{win_id} ({split_cfg['test_start']} to {split_cfg['test_end']})",
        save_path=os.path.join(RESULTS_DIR, f"exp1_nav_w{win_id}.png"),
        index=test_index,
    )

    table = generate_report_table(metrics_dict)
    table.to_csv(os.path.join(RESULTS_DIR, f"exp1_metrics_w{win_id}.csv"))
    display(table)

# Average over windows
print(f"\n{'='*60}")
print("Exp1 Average over W1+W2 (mean across seeds):")
print(f"{'='*60}")
avg_all = {}
for rt, mlist in exp1_all_metrics.items():
    if mlist:
        avg_all[REWARD_LABELS[rt]] = {k: round(float(np.mean([m[k] for m in mlist])), 3) for k in mlist[0]}
avg_table = generate_report_table(avg_all)
avg_table.to_csv(os.path.join(RESULTS_DIR, "exp1_avg_metrics.csv"))
display(avg_table)
print("Exp1 complete.")


Window 1: test 2023-01-01 to 2024-01-01
[data_utils] Split W1: train=1095d, test=365d
  PPO-Sortino s42: Return=+33.0%  Sortino=1.272  MDD=19.4%
  PPO-Sortino s123: Return=+39.3%  Sortino=1.414  MDD=18.8%
  PPO-Sortino s777: Return=+35.2%  Sortino=1.306  MDD=21.5%
  PPO-Sharpe s42: Return=+32.6%  Sortino=1.256  MDD=19.5%
  PPO-Sharpe s123: Return=+39.0%  Sortino=1.409  MDD=18.8%
  PPO-Sharpe s777: Return=+33.9%  Sortino=1.277  MDD=21.6%
  PPO-Raw s42: Return=+33.0%  Sortino=1.273  MDD=19.4%
  PPO-Raw s123: Return=+39.1%  Sortino=1.410  MDD=18.7%
  PPO-Raw s777: Return=+35.3%  Sortino=1.307  MDD=21.4%
[plot] Saved: e:\FPTU\FPTU_SE8\REL301m\experiments\results\exp1_nav_w1.png


,PPO-Sortino,PPO-Sharpe,PPO-Raw,Equal-Weight
Metric,,,,
Total Return (%),35.837,35.167,35.783,83.52
Ann. Return (%),25.907,25.437,25.870,52.07
Sharpe Ratio,0.925,0.912,0.923,1.262
Sortino Ratio,1.331,1.314,1.330,1.818
Max Drawdown (%),19.880,19.943,19.853,28.9
Calmar Ratio,1.310,1.283,1.309,1.802
Alloc. Entropy,0.984,0.983,0.985,1.0
Trading Freq. (%),15.200,16.700,15.400,-
Total Fees (%),0.467,0.490,0.473,-



Window 2: test 2024-01-01 to 2025-01-01
[data_utils] Split W2: train=1094d, test=366d
  PPO-Sortino s42: Return=+74.4%  Sortino=1.940  MDD=24.6%
  PPO-Sortino s123: Return=+70.3%  Sortino=1.899  MDD=23.3%
  PPO-Sortino s777: Return=+73.6%  Sortino=1.929  MDD=24.1%
  PPO-Sharpe s42: Return=+74.7%  Sortino=1.947  MDD=24.6%
  PPO-Sharpe s123: Return=+68.2%  Sortino=1.859  MDD=23.4%
  PPO-Sharpe s777: Return=+73.5%  Sortino=1.926  MDD=24.1%
  PPO-Raw s42: Return=+74.6%  Sortino=1.944  MDD=24.6%
  PPO-Raw s123: Return=+65.6%  Sortino=1.807  MDD=23.5%
  PPO-Raw s777: Return=+73.8%  Sortino=1.933  MDD=24.1%
[plot] Saved: e:\FPTU\FPTU_SE8\REL301m\experiments\results\exp1_nav_w2.png


,PPO-Sortino,PPO-Sharpe,PPO-Raw,Equal-Weight
Metric,,,,
Total Return (%),72.760,72.150,71.330,88.35
Ann. Return (%),50.690,50.287,49.743,54.64
Sharpe Ratio,1.245,1.239,1.229,1.011
Sortino Ratio,1.923,1.911,1.895,1.534
Max Drawdown (%),24.020,24.030,24.090,32.74
Calmar Ratio,2.111,2.092,2.064,1.669
Alloc. Entropy,0.994,0.993,0.992,1.0
Trading Freq. (%),11.367,12.267,12.267,-
Total Fees (%),0.370,0.387,0.393,-



Exp1 Average over W1+W2 (mean across seeds):


,PPO-Sortino,PPO-Sharpe,PPO-Raw
Metric,,,
Total Return (%),54.299,53.659,53.556
Ann. Return (%),38.298,37.862,37.806
Sharpe Ratio,1.085,1.076,1.076
Sortino Ratio,1.627,1.613,1.613
Max Drawdown (%),21.950,21.986,21.971
Calmar Ratio,1.711,1.688,1.687
Alloc. Entropy,0.989,0.988,0.988
Trading Freq. (%),13.284,14.483,13.834
Total Fees (%),0.418,0.439,0.433


Exp1 complete.


## Experiment 2: PPO vs Baselines
PPO-Sortino vs Equal-Weight vs Buy&Hold BTC vs Market-Cap Weight

In [4]:
exp2_ppo_metrics = []
exp2_ew_metrics = []
exp2_bnh_metrics = []
exp2_mcap_metrics = []

for split_cfg in WALK_FORWARD_SPLITS[:2]:
    win_id = split_cfg["id"]
    print(f"\n{'='*60}")
    print(f"Window {win_id}: test {split_cfg['test_start']} to {split_cfg['test_end']}")
    print(f"{'='*60}")

    _, test_data = get_split(all_data, split_cfg)
    test_index = next(iter(test_data.values())).index

    nav_dict = {}
    metrics_dict = {}

    # PPO-Sortino (mean over seeds)
    seed_metrics = []
    seed_navs = []
    for seed in SEEDS:
        mp = find_mlp_model("sortino_style", win_id, seed)
        if mp is None:
            continue
        nav, metrics, _ = run_backtest(
            model_path=mp, test_data=test_data,
            reward_type="sortino_style", use_cash=True, lambda_risk=0.3,
        )
        seed_metrics.append(metrics)
        seed_navs.append(nav)
        print(f"  PPO-Sortino s{seed}: Return={metrics['total_return']:+.1f}%  "
              f"Sortino={metrics['sortino']:.3f}  MDD={metrics['max_drawdown']:.1f}%")

    if seed_metrics:
        metrics_dict["PPO-Sortino"] = avg_seeds(seed_metrics)
        exp2_ppo_metrics.append(metrics_dict["PPO-Sortino"])
        min_len = min(len(n) for n in seed_navs)
        nav_dict["PPO-Sortino"] = np.stack([n[:min_len] for n in seed_navs]).mean(axis=0)

    # Baselines
    nav_ew, met_ew = run_equal_weight_baseline(test_data)
    nav_bnh, met_bnh = run_bnh_baseline(test_data, ticker="BTC")
    nav_mcap, met_mcap = run_mcap_baseline(test_data)

    nav_dict["Equal-Weight"] = nav_ew
    nav_dict["B&H BTC"] = nav_bnh
    nav_dict["MktCap-Weight"] = nav_mcap
    metrics_dict["Equal-Weight"] = met_ew
    metrics_dict["B&H BTC"] = met_bnh
    metrics_dict["MktCap-Weight"] = met_mcap
    exp2_ew_metrics.append(met_ew)
    exp2_bnh_metrics.append(met_bnh)
    exp2_mcap_metrics.append(met_mcap)

    print(f"  Equal-Weight:  Return={met_ew['total_return']:+.1f}%  Sortino={met_ew['sortino']:.3f}  MDD={met_ew['max_drawdown']:.1f}%")
    print(f"  B&H BTC:       Return={met_bnh['total_return']:+.1f}%  Sortino={met_bnh['sortino']:.3f}  MDD={met_bnh['max_drawdown']:.1f}%")
    print(f"  MktCap-Weight: Return={met_mcap['total_return']:+.1f}%  Sortino={met_mcap['sortino']:.3f}  MDD={met_mcap['max_drawdown']:.1f}%")

    # Plot
    plot_nav_comparison(
        nav_dict=nav_dict,
        title=f"Exp2: PPO vs Baselines - W{win_id} ({split_cfg['test_start']} to {split_cfg['test_end']})",
        save_path=os.path.join(RESULTS_DIR, f"exp2_nav_w{win_id}.png"),
        index=test_index,
    )

    table = generate_report_table(metrics_dict)
    table.to_csv(os.path.join(RESULTS_DIR, f"exp2_metrics_w{win_id}.csv"))
    display(table)

# Average
print(f"\n{'='*60}")
print("Exp2 Average over W1+W2:")
print(f"{'='*60}")
avg_all = {}
for lbl, mlist in [("PPO-Sortino", exp2_ppo_metrics), ("Equal-Weight", exp2_ew_metrics),
                    ("B&H BTC", exp2_bnh_metrics), ("MktCap-Weight", exp2_mcap_metrics)]:
    if mlist:
        avg_all[lbl] = {k: round(float(np.mean([m[k] for m in mlist])), 3) for k in mlist[0]}
avg_table = generate_report_table(avg_all)
avg_table.to_csv(os.path.join(RESULTS_DIR, "exp2_avg_metrics.csv"))
display(avg_table)
print("Exp2 complete.")


Window 1: test 2023-01-01 to 2024-01-01
[data_utils] Split W1: train=1095d, test=365d
  PPO-Sortino s42: Return=+33.0%  Sortino=1.272  MDD=19.4%
  PPO-Sortino s123: Return=+39.3%  Sortino=1.414  MDD=18.8%
  PPO-Sortino s777: Return=+35.2%  Sortino=1.306  MDD=21.5%
  Equal-Weight:  Return=+83.5%  Sortino=1.818  MDD=28.9%
  B&H BTC:       Return=+154.2%  Sortino=2.924  MDD=20.1%
  MktCap-Weight: Return=+117.5%  Sortino=2.403  MDD=22.7%
[plot] Saved: e:\FPTU\FPTU_SE8\REL301m\experiments\results\exp2_nav_w1.png


,PPO-Sortino,Equal-Weight,B&H BTC,MktCap-Weight
Metric,,,,
Total Return (%),35.837,83.52,154.23,117.5
Ann. Return (%),25.907,52.07,90.44,70.99
Sharpe Ratio,0.925,1.262,1.796,1.563
Sortino Ratio,1.331,1.818,2.924,2.403
Max Drawdown (%),19.880,28.9,20.06,22.73
Calmar Ratio,1.310,1.802,4.509,3.123
Alloc. Entropy,0.984,1.0,0.0,0.817
Trading Freq. (%),15.200,-,-,-
Total Fees (%),0.467,-,-,-



Window 2: test 2024-01-01 to 2025-01-01
[data_utils] Split W2: train=1094d, test=366d
  PPO-Sortino s42: Return=+74.4%  Sortino=1.940  MDD=24.6%
  PPO-Sortino s123: Return=+70.3%  Sortino=1.899  MDD=23.3%
  PPO-Sortino s777: Return=+73.6%  Sortino=1.929  MDD=24.1%
  Equal-Weight:  Return=+88.3%  Sortino=1.534  MDD=32.7%
  B&H BTC:       Return=+111.5%  Sortino=1.878  MDD=26.2%
  MktCap-Weight: Return=+88.5%  Sortino=1.541  MDD=31.9%
[plot] Saved: e:\FPTU\FPTU_SE8\REL301m\experiments\results\exp2_nav_w2.png


,PPO-Sortino,Equal-Weight,B&H BTC,MktCap-Weight
Metric,,,,
Total Return (%),72.760,88.35,111.53,88.55
Ann. Return (%),50.690,54.64,67.51,54.75
Sharpe Ratio,1.245,1.011,1.175,1.0
Sortino Ratio,1.923,1.534,1.878,1.541
Max Drawdown (%),24.020,32.74,26.18,31.93
Calmar Ratio,2.111,1.669,2.578,1.715
Alloc. Entropy,0.994,1.0,0.0,0.817
Trading Freq. (%),11.367,-,-,-
Total Fees (%),0.370,-,-,-



Exp2 Average over W1+W2:


,PPO-Sortino,Equal-Weight,B&H BTC,MktCap-Weight
Metric,,,,
Total Return (%),54.299,85.935,132.88,103.025
Ann. Return (%),38.298,53.355,78.975,62.87
Sharpe Ratio,1.085,1.136,1.486,1.281
Sortino Ratio,1.627,1.676,2.401,1.972
Max Drawdown (%),21.950,30.82,23.12,27.33
Calmar Ratio,1.711,1.736,3.543,2.419
Alloc. Entropy,0.989,1.0,0.0,0.817
Trading Freq. (%),13.284,-,-,-
Total Fees (%),0.418,-,-,-


Exp2 complete.


## Experiment 3: MLP vs LSTM Policy Comparison
Both use sortino_style reward, use_cash=True

In [5]:
CONDITIONS = [
    ("PPO-MLP",  "mlp",  False),
    ("PPO-LSTM", "lstm", True),
]
REWARD_TYPE = "sortino_style"

exp3_window_avg = {}

for split_cfg in WALK_FORWARD_SPLITS[:2]:
    win_id = split_cfg["id"]
    print(f"\n{'='*60}")
    print(f"Window {win_id}: test {split_cfg['test_start']} to {split_cfg['test_end']}")
    print(f"{'='*60}")

    _, test_data = get_split(all_data, split_cfg)
    test_index = next(iter(test_data.values())).index

    nav_dict = {}
    avg_metrics = {}

    for label, policy_tag, use_lstm in CONDITIONS:
        seed_results = []
        for seed in SEEDS:
            if policy_tag == "lstm":
                mp = find_lstm_model(REWARD_TYPE, win_id, seed)
            else:
                mp = find_mlp_model(REWARD_TYPE, win_id, seed)
            if mp is None:
                print(f"  [WARN] {label} seed={seed} not found")
                continue
            try:
                nav, metrics, wlog = run_backtest(
                    model_path=mp, test_data=test_data,
                    reward_type=REWARD_TYPE, use_cash=True,
                    use_lstm=use_lstm, lambda_risk=0.3,
                )
                seed_results.append((nav, metrics, wlog))
                print(f"  {label} s{seed}: Return={metrics['total_return']:+.1f}%  "
                      f"Sortino={metrics['sortino']:+.3f}  MDD={metrics['max_drawdown']:.1f}%")
            except Exception as ex:
                print(f"  [WARN] {label} seed={seed} failed: {ex}")

        if not seed_results:
            continue

        avg_metrics[label] = avg_seeds([r[1] for r in seed_results])
        min_len = min(len(r[0]) for r in seed_results)
        nav_dict[label] = np.stack([r[0][:min_len] for r in seed_results]).mean(axis=0)

    # Baselines
    nav_ew, met_ew = run_equal_weight_baseline(test_data)
    nav_bnh, met_bnh = run_bnh_baseline(test_data, ticker="BTC")
    nav_dict["Equal-Weight"] = nav_ew
    nav_dict["B&H BTC"] = nav_bnh
    avg_metrics["Equal-Weight"] = met_ew
    avg_metrics["B&H BTC"] = met_bnh

    exp3_window_avg[win_id] = avg_metrics

    plot_nav_comparison(
        nav_dict=nav_dict,
        title=f"Exp3: MLP vs LSTM - W{win_id} ({split_cfg['test_start']} to {split_cfg['test_end']})",
        save_path=os.path.join(RESULTS_DIR, f"exp3_nav_w{win_id}.png"),
        index=test_index,
    )

    table = generate_report_table(avg_metrics)
    table.to_csv(os.path.join(RESULTS_DIR, f"exp3_metrics_w{win_id}.csv"))
    display(table)

# Grand average
print(f"\n{'='*60}")
print("Exp3 Average over W1+W2:")
print(f"{'='*60}")
labels = list(next(iter(exp3_window_avg.values())).keys())
grand_avg = {}
for label in labels:
    per_win = [exp3_window_avg[w][label] for w in exp3_window_avg if label in exp3_window_avg[w]]
    if per_win:
        grand_avg[label] = {k: round(float(np.mean([m[k] for m in per_win])), 3) for k in per_win[0]}

avg_table = generate_report_table(grand_avg)
avg_table.to_csv(os.path.join(RESULTS_DIR, "exp3_avg_metrics.csv"))
display(avg_table)

print("\nExp3 Summary (avg W1+W2):")
for label in labels:
    if label not in grand_avg:
        continue
    m = grand_avg[label]
    print(f"  {label:20s}  Return={m['total_return']:+6.1f}%  "
          f"Sortino={m['sortino']:+.3f}  MDD={m['max_drawdown']:.1f}%  "
          f"Calmar={m['calmar']:.3f}")
print("\nExp3 complete.")


Window 1: test 2023-01-01 to 2024-01-01
[data_utils] Split W1: train=1095d, test=365d
  PPO-MLP s42: Return=+33.0%  Sortino=+1.272  MDD=19.4%
  PPO-MLP s123: Return=+39.3%  Sortino=+1.414  MDD=18.8%
  PPO-MLP s777: Return=+35.2%  Sortino=+1.306  MDD=21.5%
  PPO-LSTM s42: Return=+37.2%  Sortino=+1.462  MDD=20.3%
  PPO-LSTM s123: Return=+34.3%  Sortino=+1.281  MDD=21.9%
  PPO-LSTM s777: Return=+38.3%  Sortino=+1.439  MDD=20.0%
[plot] Saved: e:\FPTU\FPTU_SE8\REL301m\experiments\results\exp3_nav_w1.png


,PPO-MLP,PPO-LSTM,Equal-Weight,B&H BTC
Metric,,,,
Total Return (%),35.837,36.613,83.52,154.23
Ann. Return (%),25.907,26.447,52.07,90.44
Sharpe Ratio,0.925,0.962,1.262,1.796
Sortino Ratio,1.331,1.394,1.818,2.924
Max Drawdown (%),19.880,20.710,28.9,20.06
Calmar Ratio,1.310,1.281,1.802,4.509
Alloc. Entropy,0.984,0.977,1.0,0.0
Trading Freq. (%),15.200,17.700,-,-
Total Fees (%),0.467,0.503,-,-



Window 2: test 2024-01-01 to 2025-01-01
[data_utils] Split W2: train=1094d, test=366d
  PPO-MLP s42: Return=+74.4%  Sortino=+1.940  MDD=24.6%
  PPO-MLP s123: Return=+70.3%  Sortino=+1.899  MDD=23.3%
  PPO-MLP s777: Return=+73.6%  Sortino=+1.929  MDD=24.1%
  PPO-LSTM s42: Return=+66.1%  Sortino=+1.927  MDD=25.6%
  PPO-LSTM s123: Return=+72.9%  Sortino=+1.931  MDD=24.4%
  PPO-LSTM s777: Return=+70.6%  Sortino=+1.889  MDD=24.3%
[plot] Saved: e:\FPTU\FPTU_SE8\REL301m\experiments\results\exp3_nav_w2.png


,PPO-MLP,PPO-LSTM,Equal-Weight,B&H BTC
Metric,,,,
Total Return (%),72.760,69.867,88.35,111.53
Ann. Return (%),50.690,48.790,54.64,67.51
Sharpe Ratio,1.245,1.239,1.011,1.175
Sortino Ratio,1.923,1.916,1.534,1.878
Max Drawdown (%),24.020,24.800,32.74,26.18
Calmar Ratio,2.111,1.970,1.669,2.578
Alloc. Entropy,0.994,0.988,1.0,0.0
Trading Freq. (%),11.367,12.067,-,-
Total Fees (%),0.370,0.380,-,-



Exp3 Average over W1+W2:


,PPO-MLP,PPO-LSTM,Equal-Weight,B&H BTC
Metric,,,,
Total Return (%),54.299,53.240,85.935,132.88
Ann. Return (%),38.298,37.618,53.355,78.975
Sharpe Ratio,1.085,1.101,1.136,1.486
Sortino Ratio,1.627,1.655,1.676,2.401
Max Drawdown (%),21.950,22.755,30.82,23.12
Calmar Ratio,1.711,1.625,1.736,3.543
Alloc. Entropy,0.989,0.982,1.0,0.0
Trading Freq. (%),13.284,14.883,-,-
Total Fees (%),0.418,0.442,-,-



Exp3 Summary (avg W1+W2):
  PPO-MLP               Return= +54.3%  Sortino=+1.627  MDD=21.9%  Calmar=1.711
  PPO-LSTM              Return= +53.2%  Sortino=+1.655  MDD=22.8%  Calmar=1.625
  Equal-Weight          Return= +85.9%  Sortino=+1.676  MDD=30.8%  Calmar=1.736
  B&H BTC               Return=+132.9%  Sortino=+2.401  MDD=23.1%  Calmar=3.543

Exp3 complete.


## Summary

In [6]:
print("All experiments complete.")
print(f"Results saved in: {os.path.abspath(RESULTS_DIR)}")
print("\nOutput files:")
for f in sorted(os.listdir(RESULTS_DIR)):
    print(f"  {f}")

All experiments complete.
Results saved in: e:\FPTU\FPTU_SE8\REL301m\experiments\results

Output files:
  eda
  exp1_avg_metrics.csv
  exp1_metrics_w1.csv
  exp1_metrics_w2.csv
  exp1_nav_w1.png
  exp1_nav_w2.png
  exp2_avg_metrics.csv
  exp2_metrics_w1.csv
  exp2_metrics_w2.csv
  exp2_nav_w1.png
  exp2_nav_w2.png
  exp3_avg_metrics.csv
  exp3_metrics_w1.csv
  exp3_metrics_w2.csv
  exp3_nav_w1.png
  exp3_nav_w2.png
